# 02 Normalization

Load the demultiplexed singlet objects from notebook 01b, preserve raw counts in `layers["counts"]`, normalize/log-transform expression, identify highly variable genes, scale, run PCA, and save one normalized object per dataset for notebook 03.


In [ ]:
import sys
sys.path.append("/home/dk5299/Projects_31926/RNA-seq/Kallies_Sci_Immuno_2025")

import pandas as pd
import scanpy as sc

from src.paths import INTERMEDIATE_DIR, RESULTS_DIR

sc.settings.verbosity = 3
sc.settings.set_figure_params(dpi=100, facecolor="white")


## Input and Output Files

Notebook 01b writes the demultiplexed singlet object and the three sample-group subsets. Notebook 03 expects the matching `02_normalized_hvg_pca*.h5ad` outputs.


In [ ]:
datasets = {
    "all": {
        "input": INTERMEDIATE_DIR / "01b_demultiplexed_singlets.h5ad",
        "output": INTERMEDIATE_DIR / "02_normalized_hvg_pca.h5ad",
    },
    "sIEL": {
        "input": INTERMEDIATE_DIR / "01b_sIEL.h5ad",
        "output": INTERMEDIATE_DIR / "02_normalized_hvg_pca_sIEL.h5ad",
    },
    "sLPL": {
        "input": INTERMEDIATE_DIR / "01b_sLPL.h5ad",
        "output": INTERMEDIATE_DIR / "02_normalized_hvg_pca_sLPL.h5ad",
    },
    "rechallenge": {
        "input": INTERMEDIATE_DIR / "01b_rechallenge.h5ad",
        "output": INTERMEDIATE_DIR / "02_normalized_hvg_pca_rechallenge.h5ad",
    },
}

pd.DataFrame(
    [{"dataset": name, "input": paths["input"], "output": paths["output"]} for name, paths in datasets.items()]
)


## Normalization Parameters


In [ ]:
target_sum = 1e4
n_top_genes = 3000
n_pcs = 50
random_state = 0
hvg_flavor = "seurat"


## Normalize, Find HVGs, and Run PCA

`.X` is transformed to log-normalized expression and then scaled for PCA. Raw counts are retained in `layers["counts"]`. Full-gene log-normalized expression is stored in `.raw` before scaling so marker plots and ranking can use `use_raw=True`.


In [ ]:
def normalize_hvg_pca(dataset_name, input_file, output_file):
    print(f"\n--- {dataset_name} ---")
    print("Loading:", input_file)
    adata = sc.read_h5ad(input_file)
    print(adata)

    if "counts" not in adata.layers:
        adata.layers["counts"] = adata.X.copy()

    adata.X = adata.layers["counts"].copy()
    adata.uns.pop("log1p", None)

    sc.pp.normalize_total(adata, target_sum=target_sum)
    sc.pp.log1p(adata)

    adata.raw = adata.copy()

    hvg_kwargs = {
        "n_top_genes": n_top_genes,
        "flavor": hvg_flavor,
    }
    if dataset_name == "all" and "sample_group" in adata.obs:
        hvg_kwargs["batch_key"] = "sample_group"

    sc.pp.highly_variable_genes(adata, **hvg_kwargs)
    print("Highly variable genes:", int(adata.var["highly_variable"].sum()))

    sc.pp.scale(adata, max_value=10)
    sc.tl.pca(
        adata,
        n_comps=n_pcs,
        use_highly_variable=True,
        svd_solver="arpack",
        random_state=random_state,
    )

    output_file.parent.mkdir(parents=True, exist_ok=True)
    adata.write_h5ad(output_file)
    print("Saved:", output_file)

    return adata


In [ ]:
normalized = {}

for dataset_name, paths in datasets.items():
    normalized[dataset_name] = normalize_hvg_pca(
        dataset_name=dataset_name,
        input_file=paths["input"],
        output_file=paths["output"],
    )


## QC Summaries

Review dimensions, sample composition, and HVG counts before proceeding to clustering.


In [ ]:
summary_rows = []

for dataset_name, adata in normalized.items():
    row = {
        "dataset": dataset_name,
        "n_cells": adata.n_obs,
        "n_genes": adata.n_vars,
        "n_hvg": int(adata.var["highly_variable"].sum()),
        "has_counts_layer": "counts" in adata.layers,
        "has_raw_lognorm": adata.raw is not None,
        "n_pcs": adata.obsm["X_pca"].shape[1],
    }
    summary_rows.append(row)

normalization_summary = pd.DataFrame(summary_rows)
normalization_summary


In [ ]:
summary_file = RESULTS_DIR / "02_normalization_summary.csv"
summary_file.parent.mkdir(parents=True, exist_ok=True)
normalization_summary.to_csv(summary_file, index=False)
print("Saved:", summary_file)


In [ ]:
if "all" in normalized and "sample_group" in normalized["all"].obs:
    pd.crosstab(normalized["all"].obs["sample_group"], normalized["all"].obs["hash_id"] if "hash_id" in normalized["all"].obs else normalized["all"].obs["sample_group"])


## Optional HVG Plot


In [ ]:
sc.pl.highly_variable_genes(normalized["all"])
